# Django Relationships & ORM

## ManyToManyField

A many-to-many relationship allows each record on both sides to relate to multiple records on the other:

```python
class Book(models.Model):
    title = models.CharField(max_length=200)
    authors = models.ManyToManyField('Author', blank=True)
```

Managing the relationship in the shell:
```python
book = Book.objects.get(pk=1)
author = Author.objects.get(pk=2)

book.authors.add(author)
book.authors.remove(author)
book.authors.all()
```


## Django Shell: CRUD

```bash
python manage.py shell
```

```python
from library.models import Book, Author

# Create
author = Author.objects.create(name='Alice')
book = Book.objects.create(title='Django Unleashed')
book.authors.add(author)

# Read
Book.objects.all()
book.authors.all()

# Update
book.title = 'Django Deep Dive'
book.save()

# Delete
book.delete()
```


## CRUD via Views

You can expose model CRUD through Django views and test them with tools like Postman or the browsable API.

```python
from django.http import JsonResponse
from .models import Book

def book_list(request):
    books = list(Book.objects.values('id', 'title'))
    return JsonResponse(books, safe=False)
```


## Types of Model Inheritance

### Concrete (Simple) Inheritance
The child model gets its own database table and inherits the parent's fields. Use when extending a third-party model like `User`:
```python
from django.contrib.auth.models import User

class Customer(User):
    loyalty_points = models.IntegerField(default=0)
```

### Abstract Base Class
No database table is created for the parent. Common fields are shared across child models:
```python
class TimestampMixin(models.Model):
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        abstract = True

class Article(TimestampMixin):
    title = models.CharField(max_length=200)
```

### Proxy Model
Same database table as the parent, but with customized Python behavior (e.g., a custom manager or ordering):
```python
class PublishedArticleManager(models.Manager):
    def get_queryset(self):
        return super().get_queryset().filter(status='published')

class PublishedArticle(Article):
    objects = PublishedArticleManager()

    class Meta:
        proxy = True
```


## Advanced Query Filters

```python
from library.models import Book

# __in — field value is in a list
Book.objects.filter(author__in=[1, 2, 3])

# __isnull — field is NULL
Book.objects.filter(publisher__isnull=True)

# __gt — field value is greater than
Book.objects.filter(price__gt=20)

# exclude + first
Book.objects.exclude(status='draft').first()
```


## Django ORM Tools Overview

Django provides three powerful tools for aggregating and filtering data at the database level:

- **Aggregation** — compute summary values (count, sum, average) across a queryset
- **Annotation** — add computed columns to each row in a queryset
- **Q Objects** — combine multiple filter conditions with AND (`&`) and OR (`|`)

These are covered in detail in the next session.


## Summary

- `ManyToManyField` links records on both sides with a join table managed by Django.
- Three inheritance styles: concrete (own table), abstract (no table), proxy (same table, custom behavior).
- Proxy models are ideal for adding custom managers without changing the schema.
- Field lookups: `__in`, `__isnull`, `__gt`, and many more — chain them inside `filter()` or `exclude()`.
